### 1. Basic Tasks

In [0]:
# 1.
data = [
    {"id": 101, "name": "Amit", "age": 20, "course": "BCA", "marks": 78},
    {"id": 102, "name": "Riya", "age": 21, "course": "BBA", "marks": 85},
    {"id": 103, "name": "Raj", "age": 19, "course": "BCA", "marks": 72},
    {"id": 104, "name": "Neha", "age": 22, "course": "MCA", "marks": 91},
    {"id": 105, "name": "Karan", "age": 20, "course": "BBA", "marks": 68},
    {"id": 106, "name": "Priya", "age": 21, "course": "MCA", "marks": 88}
]

df = spark.createDataFrame(data)

In [0]:
df.display()

In [0]:
# 2.
df = spark.read.csv("/Volumes/cyntexa_dev/sales/raw/sales.csv", header = True, inferSchema = True)
df.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema = StructType([
    StructField("order_id", IntegerType(), nullable = True),
    StructField("customer_id", IntegerType(), nullable = True),
    StructField("transaction_id", IntegerType(), nullable = True),
    StructField("product_id", IntegerType(), nullable = True),
    StructField("quantity", IntegerType(), nullable = True),
    StructField("discount_amount", DoubleType(), nullable = True),
    StructField("total_amount", DoubleType(), nullable = True),
    StructField("order_date", DateType(), nullable = True)
])

df1 = spark.read.csv("/Volumes/cyntexa_dev/sales/raw/sales.csv", header = True, schema = schema)
df1.printSchema()

In [0]:
df.schema == df1.schema

In [0]:
from pyspark.sql.functions import *

In [0]:
# 3.
df.withColumn("total_amount", col("total_amount")*1.5)

In [0]:
df.show()

The data transfoemation operation do not immediately execute in spark, instead it creates an execution plan describing the operations that needs to be performed 

when an action such as show() if called spark triggers the execution plan and process the data. 
This type of evaluation is called lazy evaluation

### 2. Intermediate Tasks

In [0]:
# End to end ETL
sales_df = spark.read.csv("/Volumes/cyntexa_dev/sales/raw/sales.csv", header = True, inferSchema = True)
sales_df.printSchema()

In [0]:
sales_df_filtered = sales_df.filter(col("quantity") > 1) \
    .withColumn("File_path", sales_df['_metadata.file_path'])

In [0]:
sales_df_filtered.write.mode("overwrite").saveAsTable("cyntexa_dev.sales.cleaned_sales")

In [0]:
%sql 
select * from cyntexa_dev.sales.cleaned_sales

In [0]:
# 5.
df = spark.read.json("/Volumes/cyntexa_dev/sales/raw/drivers.json")
df.printSchema()

In [0]:
df_flattened = df.select("code","dob","driverId","driverRef",
                         col("name.forename").alias("firstname"),
                         col("name.surname").alias("lastname"),
                         "nationality","number","url")
df_flattened.printSchema()

In [0]:
# 6.
df = df_flattened.filter(col("nationality").isNotNull()) \
    .withColumn("full_name", concat(col("firstname"), lit(" "), col("lastname"  ))) \
        .select("driverId","full_name","nationality") \
            .groupBy("nationality") \
                .agg(count("driverId").alias("driver_count"))

In [0]:
df.explain()

### 3. Advanced Tasks

In [0]:
# 7.
import pandas as pd
df = pd.read_csv("/Volumes/cyntexa_dev/sales/raw/sales.csv")
df.head()

In [0]:
df.dropna()
df["price"] = df["total_amount"] / df["quantity"]
df["net_amount"] = df["total_amount"] - df["discount_amount"]

df.head()

In [0]:
# spark approach
df = spark.read.csv("/Volumes/cyntexa_dev/sales/raw/sales.csv", header=True, inferSchema=True)
df = df.dropna() \
    .withColumn("price", col("total_amount") / col("quantity")) \
        .withColumn("net_amount", col("total_amount") - col("discount_amount"))
df.display()

Pandas vs PySpark
- Pandas loads the datasets into memory of a single machine. A very large dataset may exceed the available RAM. Spark divided the data innot partitions and processes them accross multiple executer in a distributive manner

- Aggregations such as group by can be very expensive when dataset is very large when working on a single machine, Spark distributes the aggregation accross a cluster

- Pandas is designed for processing data on one machine while spark is designed for distributive processing. Spark can process large datasets by distributing the workload accross multiple machines.

Pandas is more suitable for small datasets for local analytics, While Spark is more suitable for large-scale e-commerce data processing 

In [0]:
# 8.
sales_df = spark.read.csv("/Volumes/cyntexa_dev/sales/raw/sales.csv",
header = True,
inferSchema = True)

sales_df.write.mode("overwrite").partitionBy("order_date").saveAsTable("cyntexa_dev.sales.sales_partitioned")

In [0]:
result = spark.read.table("cyntexa_dev.sales.sales_partitioned") \
    .filter(col("order_date") >= "2023-12-03")
result.show()

Spark uses lazy evaluation, so the filter is not executed immediately. When an action
such as show() or count() is called, Spark creates and executes the physical plan.
The physical plan can use the date filter as a partition filter, allowing Spark to
skip irrelevant date partitions.

In [0]:
# 9.
# Revenue by each customer

result = sales_df.groupBy("customer_id").agg(sum("total_amount").alias("total_revenue")).orderBy(col("total_revenue").desc())
result.show()

In [0]:
result.write.mode("overwrite").saveAsTable("cyntexa_dev.sales.customer_revenue")